# 00 · The question, available data, and the first decision

**Can we remove tracker errors without deleting movements that really
happened?** This notebook establishes what data and public-model inputs
are available. The first scientific result is a preservation-versus-repair
comparison, not binary gait classification.

Run each notebook in a fresh kernel, in order **00 → 04**. Notebook 05 is
an external visual stress test. These are offline restoration experiments:
the model may inspect the declared complete clip. They do not claim causal
forecasting or clinical diagnosis.

**The default is real data.** Export `MP_RUN_ROOT`, the AMASS and GAVD data
paths, and the model configuration before opening Jupyter. See the
[launch guide](../../../../../slurm/motion-preservation/README.md).
For a CPU walkthrough of the mechanics, explicitly choose `MP_MODE=demo`
and a separate run directory. Demo outputs cannot establish a research result.

[Proposal](../../../../../docs/studies/motion-preservation/protocol/proposal.md)
· [Notebook guide](../../../../../notebooks/motion_preservation/README.md)

In [1]:
from pathlib import Path
import json
import os
import sys
from time import perf_counter

project_override = os.environ.get("GAVD6_ROOT")
candidates = ([Path(project_override).expanduser()] if project_override else
              [Path.cwd(), *Path.cwd().parents])
PROJECT_ROOT = next((p.resolve() for p in candidates
                     if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the gavd6 checkout.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)  # Resolve manifest/config paths from the checkout in every kernel.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, Video, display
from gavd6_sjepa.research_directions.motion_preservation import workflow, plots

get_ipython().run_line_magic("matplotlib", "inline")
plt.rcParams.update({"figure.figsize": (9, 3.5), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
cfg = workflow.config_from_environment()
RUN_ROOT = Path(cfg.run_root)
print(f"Mode: {cfg.mode}; run directory: {RUN_ROOT}")
if cfg.mode == "demo":
    display(Markdown("**DEMO ONLY: generated fixtures and stand-in models. "
                     "These outputs are not evidence about AMASS, GAVD, or a pretrained prior.**"))

Mode: real; run directory: /hai/scratch/tedmui/alexpose/experiments/sjepa/gavd6/outputs/motion-preservation/pilot-01


## 1. Why ordinary reconstruction error is not enough

Suppose a real ankle excursion lasts only a few frames. Removing that
excursion may slightly reduce the average error across the entire body,
even though the useful movement information was lost.

We need two separate measurements:

- **Repair:** how much squared error at observed joints is removed?
- **Preservation:** how much of the known real movement remains?

Filling missing joints is measured separately. An interpolated gap is
a model input, not an observation that can earn tracking-repair credit.

We compare methods at a repair strength chosen on separate calibration
people. A method cannot win by leaving every noisy coordinate unchanged.

In [2]:
# This table explains the design; it contains no measured results.
display(pd.DataFrame([
    {"True event": False, "Tracking noise": False, "Desired behavior": "Keep the clean movement"},
    {"True event": True,  "Tracking noise": False, "Desired behavior": "Keep the event"},
    {"True event": False, "Tracking noise": True,  "Desired behavior": "Remove the error"},
    {"True event": True,  "Tracking noise": True,  "Desired behavior": "Preserve the event and remove the error"},
]))

,True event,Tracking noise,Desired behavior
0,False,False,Keep the clean movement
1,True,False,Keep the event
2,False,True,Remove the error
3,True,True,Preserve the event and remove the error


## 2. Read the existing AMASS and GAVD manifests

AMASS supplies known body motion for controlled experiments. We use
whole-body joints so arms and trunk can carry a true event. GAVD supplies
in-the-wild videos for external inspection. Its estimated trajectories
are not 3D reference truth.

Inventory counts come from the repository manifests. Availability comes
from the configured filesystem on this machine. A manifest entry does
not imply that its raw file is available locally.

In [3]:
started = perf_counter()
inventory = workflow.inventory(cfg)
for name in ("amass", "gavd", "availability"):
    table = inventory[name]
    display(Markdown(f"### {name.capitalize()} ({len(table):,} rows)"))
    display(table.head(12))
print(f"Inventory completed in {perf_counter() - started:.1f} seconds.")

### Amass (8,854 rows)

,relative_path,source_dataset,parent_path,subject_id_candidate,motion_id,sha256,npz_keys,num_frames,pose_width,trans_frames,...,mocap_framerate,gender,status,error,person_id,original_split,role,raw_path,available,duration_s
0,BioMotionLab_NTroje/rub044/0000_treadmill_norm...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub044,0000_treadmill_norm,a1629a596af402d418585c7aec0d1ae57c4c8c5fd3c89a...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",3182,156,3182,...,120.0,male,ok,NaN,BioMotionLab_NTroje::rub044,validation,calibration,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,26.508333
1,BioMotionLab_NTroje/rub044/0001_treadmill_fast...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub044,0001_treadmill_fast,e8576e8799b2d5be54d752df71ead342ee87ddc6aa3d92...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",2833,156,2833,...,120.0,male,ok,NaN,BioMotionLab_NTroje::rub044,validation,calibration,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,23.600000
2,BioMotionLab_NTroje/rub044/0002_treadmill_slow...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub044,0002_treadmill_slow,7b92530ae5aec450baa075bf41da50a406e70b7b51b9cf...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",2792,156,2792,...,120.0,male,ok,NaN,BioMotionLab_NTroje::rub044,validation,calibration,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,23.258333
3,BioMotionLab_NTroje/rub044/0003_treadmill_jog_...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub044,0003_treadmill_jog,6950e2f140d822d86f92a8b12f9fc7ab16366afd465fcd...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",2692,156,2692,...,120.0,male,ok,NaN,BioMotionLab_NTroje::rub044,validation,calibration,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,22.425000
4,BioMotionLab_NTroje/rub044/0004_motorcycle_pos...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub044,0004_motorcycle,bfde7f928876d1b466627fda0996681a6309a70218fdfa...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",370,156,370,...,120.0,male,ok,NaN,BioMotionLab_NTroje::rub044,validation,calibration,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,3.075000
5,BioMotionLab_NTroje/rub044/0005_normal_walk1_p...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub044,0005_normal_walk1,c3bf9e05220fbbaf2a49fe43c034d01513947260e5ccaa...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",422,156,422,...,120.0,male,ok,NaN,BioMotionLab_NTroje::rub044,validation,calibration,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,3.508333
6,BioMotionLab_NTroje/rub044/0006_normal_walk2_p...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub044,0006_normal_walk2,b9bf72f9b88bab7cc0902a98b34a32be53e6bc03e4bb20...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",387,156,387,...,120.0,male,ok,NaN,BioMotionLab_NTroje::rub044,validation,calibration,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,3.216667
7,BioMotionLab_NTroje/rub044/0007_normal_walk3_p...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub044,0007_normal_walk3,702a3efa2cc8a35477f82482379d148a1844a3d3a72f93...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",369,156,369,...,120.0,male,ok,NaN,BioMotionLab_NTroje::rub044,validation,calibration,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,3.066667
8,BioMotionLab_NTroje/rub044/0008_normal_walk4_p...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub044,0008_normal_walk4,de2248ccc394a892e963531f7d044390195e2111f27a02...,"[""betas"", ""dmpls"", ""gender"", ""mocap_framerate""...",445,156,445,...,120.0,male,ok,NaN,BioMotionLab_NTroje::rub044,validation,calibration,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,3.700000
9,BioMotionLab_NTroje/rub044/0009_normal_jog1_po...,BioMotionLab_NTroje,BioMotionLab_NTroje/rub044,BioMotionLab_NTroje::rub

### Gavd (1,874 rows)

,sequence_id,video_id,url,first_frame,last_frame,n_annotated_frames,source_height,dataset_annotation,gait_pattern_annotation,cam_view,video_path,available,group_id,group_unit,role
0,cljan9b4p00043n6ligceanyp,B5hrxKe2nP8,https://www.youtube.com/watch?v=B5hrxKe2nP8,1757,2268,512,720,Abnormal Gait,parkinsons,right side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,B5hrxKe2nP8,recording_identity_not_known,external_observational
1,cljanb45y00083n6lmh1qhydd,B5hrxKe2nP8,https://www.youtube.com/watch?v=B5hrxKe2nP8,2532,2746,215,720,Abnormal Gait,parkinsons,left side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,B5hrxKe2nP8,recording_identity_not_known,external_observational
2,cljao8kyf000d3n6l0x9kgmav,TgkxrrhnvlM,https://www.youtube.com/watch?v=TgkxrrhnvlM,1,148,148,720,Abnormal Gait,abnormal,right side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,TgkxrrhnvlM,recording_identity_not_known,external_observational
3,cljaoak47000i3n6lsrb9rit9,TgkxrrhnvlM,https://www.youtube.com/watch?v=TgkxrrhnvlM,205,355,151,720,Abnormal Gait,abnormal,left side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,TgkxrrhnvlM,recording_identity_not_known,external_observational
4,cljaob36l000m3n6l9xokjqww,TgkxrrhnvlM,https://www.youtube.com/watch?v=TgkxrrhnvlM,382,813,432,720,Abnormal Gait,abnormal,right side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,TgkxrrhnvlM,recording_identity_not_known,external_observational
5,cljaobtd0000q3n6lutd71mer,TgkxrrhnvlM,https://www.youtube.com/watch?v=TgkxrrhnvlM,852,1342,491,720,Abnormal Gait,abnormal,left side,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,TgkxrrhnvlM,recording_identity_not_known,external_observational
6,cljaocn4e000u3n6lgnrx469h,TgkxrrhnvlM,https://www.youtube.com/watch?v=TgkxrrhnvlM,1346,1524,179,720,Abnormal Gait,abnormal,front,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,TgkxrrhnvlM,recording_identity_not_known,external_observational
7,cljaodqnu000y3n6ltaisg7tz,TgkxrrhnvlM,https://www.youtube.com/watch?v=TgkxrrhnvlM,1579,1756,178,720,Abnormal Gait,abnormal,back,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,TgkxrrhnvlM,recording_identity_not_known,external_observational
8,cljaoebv700113n6lw01p9hl3,TgkxrrhnvlM,https://www.youtube.com/watch?v=TgkxrrhnvlM,1788,2157,370,720,Abnormal Gait,abnormal,front,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,TgkxrrhnvlM,recording_identity_not_known,external_observational
9,cljaof3ba00153n6lcr8af6bi,TgkxrrhnvlM,https://www.youtube.com/watch?v=TgkxrrhnvlM,2185,2693,509,720,Abnormal Gait,abnormal,back,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,TgkxrrhnvlM,recording_identity_not_known,external_observational


### Availability (6 rows)

,asset,path,exists,experiment_mode
0,AMASS raw,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,real
1,SMPL-H and DMPL,/hai/scratch/tedmui/body_models,True,real
2,MoMask code,/hai/scratch/tedmui/vendor/momask-codes,True,real
3,MoMask checkpoint directory,/hai/scratch/tedmui/vendor/momask-codes/checkp...,True,real
4,Flow checkpoint,/hai/scratch/tedmui/vendor/SEA-RAFT/checkpoint...,True,real
5,GAVD video,/hai/scratch/tedmui/alexpose/experiments/sjepa...,True,real


Inventory completed in 2.6 seconds.


In [4]:
# Dataset/identity summaries are descriptive, not an eligibility decision.
amass = inventory["amass"]
for column in ("source_dataset", "split", "role"):
    if column in amass:
        display(amass.groupby(column, dropna=False).size().rename("manifest_rows").to_frame())
identity_column = next((c for c in ("person_id", "identity", "subject_id_candidate", "audited_subject_id")
                        if c in amass), None)
if identity_column:
    print(f"AMASS identities/groups represented: {amass[identity_column].nunique():,}")
gavd = inventory["gavd"]
if "video_id" in gavd:
    print(f"GAVD source recordings represented: {gavd.video_id.nunique():,}")

,manifest_rows
source_dataset,
ACCAD,247
BioMotionLab_NTroje,3061
EKUT,349
Eyes_Japan_Dataset,750
KIT,4232
MPI_HDM05,215


,manifest_rows
role,
calibration,269
development,500
final,868
train,7217


AMASS identities/groups represented: 189
GAVD source recordings represented: 348


## 3. Keep development and final evaluation separate

People are assigned before generating event or corruption variants. All
variants of a person stay together. Train the gate on an arm-leg timing
event. Use a foot-clearance event on development people for the first
decision. Once that result affects our choices, it is development data.
The trunk-pelvis event on final people remains unopened until notebook
04 is explicitly launched in final mode.

The current final configuration changes the event family, camera angle
and corruption mechanism together. This is a combined stress test; it
does not isolate which individual change caused a success or failure.

Audited AMASS identity determines the grouping when available. Uncertain
identities support only a weaker group-level claim. A new adaptation
split does not establish absence from a public prior's training data.

In [5]:
display(pd.DataFrame([
    ("train", "Learn the small gate", "Development event and corruption settings"),
    ("calibration", "Choose strength and decision thresholds", "Separate people; no gradient fitting"),
    ("development", "48-hour continue/stop decision", "Held people and development event family"),
    ("final", "One final held-event evaluation", "Reserved people, event, and configured nuisance condition"),
], columns=["Role", "Purpose", "Boundary"]))

,Role,Purpose,Boundary
0,train,Learn the small gate,Development event and corruption settings
1,calibration,Choose strength and decision thresholds,Separate people; no gradient fitting
2,development,48-hour continue/stop decision,Held people and development event family
3,final,One final held-event evaluation,"Reserved people, event, and configured nuisanc..."


## 4. Read the available backend configuration

A real run needs a successfully loaded released motion prior and an
image-motion estimate. The model bridge must state its representation,
coordinates, frame rate and inverse conversion. The quick synthetic
fixture has a different purpose and is always labelled demo.

Missing model weights or body-model assets are practical blockers. They
must not silently select a smoothing model and call it a pretrained prior.

In [6]:
config_path = RUN_ROOT / "config.json"
if config_path.is_file():
    saved_config = json.loads(config_path.read_text())
    display(pd.DataFrame([{"setting": key, "value": str(value)}
                          for key, value in saved_config.items()]))
else:
    print("Configuration is available in cfg; the workflow has not written config.json yet.")

,setting,value
0,run_root,/hai/scratch/tedmui/alexpose/experiments/sjepa...
1,mode,real
2,device,cuda
3,amass_manifest_dir,manifests/amass
4,amass_root,/hai/scratch/tedmui/alexpose/experiments/sjepa...
5,body_model_root,/hai/scratch/tedmui/body_models
6,dmpl_root,None
7,gavd_manifest_dir,manifests/gavd
8,gavd_video_root,/hai/scratch/tedmui/alexpose/experiments/sjepa...
9,gavd_pose_root,


## Decision before spending GPU time

Continue when the selected AMASS files, body assets, prior and flow
backend are available and there are enough distinct people to separate
fitting from evaluation. First use a small pilot. Measure actual elapsed
time before scaling to the proposal's planned 128 motion instances.

Next: [01 · Make controlled pairs](01_make_controlled_pairs.ipynb).